# 04 · Anomaly Detection — Isolation Forest for Market Stress

**Project 8 / 10**  
Unsupervised detection of return anomalies. COVID crash (2020) and 2022 rate shock as benchmark events.  
Contamination=5%, 130 anomalies detected 2015–2024.

> Author: Niraj Neupane | github.com/nirajneupane17

In [ ]:
import sys; sys.path.insert(0,"../src")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from feature_engineering import build_anomaly_features
from ml_models import AnomalyDetector
print("Libraries loaded")

In [ ]:
rets = pd.read_csv("../data/returns.csv", index_col="Date", parse_dates=True)
spy  = rets["SPY"].dropna()
print(f"Date range : {spy.index[0].date()} → {spy.index[-1].date()}")
print(f"Annual vol : {spy.std()*np.sqrt(252)*100:.1f}%")
print(f"Daily min  : {spy.min()*100:.2f}%  max: {spy.max()*100:.2f}%")

In [ ]:
feat4 = build_anomaly_features(spy, windows=[5,21])
det = AnomalyDetector(contamination=0.05, n_estimators=200)
det.fit(feat4)
anom_dates = det.anomaly_dates(feat4)
labels, scores = det.detect(feat4)
print(f"Total anomalies  : {len(anom_dates)}")
print(f"Anomaly rate     : {len(anom_dates)/len(feat4)*100:.1f}%")
print(f"\nAnomalies by year:")
print(det.annual_summary(feat4).to_string())

In [ ]:
# Crisis period capture rate
covid_days = feat4.loc["2020-02-20":"2020-04-30"].index
covid_anom = anom_dates.intersection(covid_days)
print(f"COVID crash window : {len(covid_days)} trading days")
print(f"Flagged as anomaly : {len(covid_anom)}  ({len(covid_anom)/len(covid_days)*100:.0f}% capture rate)")
shock_days = feat4.loc["2022-01-01":"2022-12-31"].index
shock_anom = anom_dates.intersection(shock_days)
print(f"\n2022 rate shock    : {len(shock_days)} trading days")
print(f"Flagged as anomaly : {len(shock_anom)}  ({len(shock_anom)/len(shock_days)*100:.0f}% capture rate)")

In [ ]:
img = plt.imread("../results/04_anomaly_detection.png")
fig, ax = plt.subplots(figsize=(15,8)); ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Score distribution
print(f"Score mean   : {scores.mean():.4f}")
print(f"Score std    : {scores.std():.4f}")
print(f"Score 5th pct: {np.percentile(scores,5):.4f}  ← detection threshold")
print(f"Score min    : {scores.min():.4f}  ← most anomalous day")